# Week 11 — Notebook 1: Introduction to Unsupervised Learning

**Difficulty:** ⭐⭐ (Beginner–Intermediate)  
**Estimated Time:** 30 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. **Contrast** supervised vs. unsupervised learning — both philosophically and mathematically.
2. **List and describe** the three main families of unsupervised tasks: clustering, dimensionality reduction, and density estimation.
3. **Explain** why evaluation is harder when there are no ground-truth labels.
4. **Load and explore** a synthetic retail dataset that mirrors the UCI Online Retail dataset — the running example for Week 11.


## Real-World Analogy: The Box of Unlabelled Receipts

Imagine you are a store manager who just received a box of **10,000 customer receipts** with no labels attached.

You want to discover natural groups — *heavy spenders*, *occasional buyers*, *coupon-hunters* — **without anyone telling you the groups exist in advance**.

You cannot ask a teacher "is this customer a heavy spender?". You only have the raw numbers: how much each customer spent, how often they visited, how long ago their last purchase was.

This is **unsupervised learning** in a nutshell:

> **Given data with no labels, find hidden structure.**

The word *unsupervised* simply means there is no supervisor (no labelled examples) telling the algorithm what the right answer looks like. The algorithm must discover patterns on its own.


## Why Does This Matter for ML?

Unsupervised learning is everywhere in real systems:

| Application | What unsupervised learning does |
|---|---|
| **Recommendation systems** | Groups users by behaviour (collaborative filtering) without explicit preference labels |
| **Anomaly detection** | Learns the "normal" distribution of network traffic; flags anything that falls outside it |
| **Pre-training / representation learning** | BERT, GPT — trained on raw text with no labels before fine-tuning |
| **Data compression** | PCA, autoencoders — find low-dimensional structure to store data efficiently |
| **Exploratory data analysis** | Clustering reveals segments before you even know what to predict |

In practice, **labelled data is expensive and scarce**. Unsupervised methods let you extract value from the vast amount of unlabelled data that most organisations already have.


In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn utilities we'll use throughout this notebook
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KernelDensity
from sklearn.metrics import silhouette_score

# Reproducibility
np.random.seed(42)

# Global plot style — consistent across the whole notebook
sns.set_theme(style='whitegrid')
print("Imports successful.")


## The Unsupervised Learning Landscape

There are three major families of unsupervised tasks. Think of them as three different questions you can ask about an unlabelled dataset:

```
Unsupervised Learning
│
├── 1. Clustering  ────────────── "Which data points belong together?"
│       Examples: K-Means, DBSCAN, Hierarchical Clustering
│
├── 2. Dimensionality Reduction ── "What is the simplest description of this data?"
│       Examples: PCA, t-SNE, UMAP, Autoencoders
│
└── 3. Density Estimation ──────── "What probability distribution generated this data?"
        Examples: KDE, GMMs, Normalising Flows, VAEs
```

These families often **overlap**: Gaussian Mixture Models (GMMs) are both a clustering method *and* a density estimator. Autoencoders do dimensionality reduction *and* can be used for anomaly detection. The taxonomy is a guide, not a strict boundary.


In [ ]:
np.random.seed(42)

# Generate a 2D dataset with 4 natural clusters
# make_blobs places Gaussian clouds at specified centres
X, y_true = make_blobs(
    n_samples=400,
    centers=4,          # 4 hidden groups
    cluster_std=0.9,    # spread within each group
    random_state=42
)

# ── Plot 1: raw data with NO colour coding (what the algorithm sees) ──────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X[:, 0], X[:, 1],
                alpha=0.6, s=30,
                color='steelblue',   # all points the same colour
                edgecolors='white', linewidths=0.3)
axes[0].set_title("What the algorithm sees\n(no labels)", fontsize=13)
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")

# ── Plot 2: same data WITH true cluster colours (ground truth revealed) ───────
palette = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']  # 4 distinct colours
for cluster_id in np.unique(y_true):
    mask = y_true == cluster_id
    axes[1].scatter(X[mask, 0], X[mask, 1],
                    alpha=0.6, s=30,
                    color=palette[cluster_id],
                    edgecolors='white', linewidths=0.3,
                    label=f'Cluster {cluster_id}')
axes[1].set_title("Ground truth revealed\n(hidden in real problems)", fontsize=13)
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")
axes[1].legend()

fig.suptitle("The Discovery Problem: Can you see the 4 groups in the left plot?",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## Supervised vs. Unsupervised: Formal Comparison

| Property | Supervised Learning | Unsupervised Learning |
|---|---|---|
| **Input** | Feature matrix **X** + label vector **y** | Feature matrix **X** only |
| **Output** | A mapping f: X → y (predictions on new data) | A structure: clusters, components, density |
| **Loss function** | Defined w.r.t. ground truth **y** (e.g. cross-entropy, MSE) | Internal objective (e.g. inertia, reconstruction error) — no y |
| **Evaluation** | Accuracy, F1, RMSE — easy because y_test exists | Silhouette score, BIC, visual inspection — harder |
| **Data requirement** | Requires labelled examples | Works on raw unlabelled data |
| **Typical goal** | Predict a specific target | Discover unknown structure |

**Key mathematical difference:**  
Supervised: minimise L(f(X), **y**)  
Unsupervised: minimise L(f(X)) — no **y** in the loss at all


In [ ]:
np.random.seed(42)

# Show the data matrix WITH labels (supervised setting)
df_supervised = pd.DataFrame(
    X[:8],                           # first 8 rows for display
    columns=['Feature_1', 'Feature_2']
)
df_supervised['Label'] = y_true[:8]  # the target column exists

# Show the data matrix WITHOUT labels (unsupervised setting)
df_unsupervised = df_supervised.drop(columns=['Label'])

print("=" * 55)
print("SUPERVISED setting — label column is present:")
print("=" * 55)
print(df_supervised.to_string(index=False))

print()
print("=" * 55)
print("UNSUPERVISED setting — label column is MISSING:")
print("=" * 55)
print(df_unsupervised.to_string(index=False))
print()
print("The algorithm must discover the groups from Feature_1 and Feature_2 alone.")


## The Three Families in Plain English

### 1. Clustering — "Find the Groups"
Assign each data point to one of k groups such that points in the same group are more similar to each other than to points in other groups.  
**Retail analogy:** Segment customers into "VIP", "Regular", "At-risk" — without being told those categories exist.

### 2. Dimensionality Reduction — "Find the Structure"
Map high-dimensional data to a lower-dimensional space while preserving as much information as possible.  
**Retail analogy:** Summarise each customer's 50 purchase-category columns into 2 "super-features" you can plot on a screen.

### 3. Density Estimation — "Find the Distribution"
Learn a probability density p(x) that describes how likely each point is under the data-generating process.  
**Retail analogy:** Model the distribution of basket sizes so that any basket costing > £500 is flagged as an anomaly (low probability).


In [ ]:
np.random.seed(42)

# ── Three algorithms, same unlabelled dataset ─────────────────────────────────

# 1. Clustering: K-Means
km = KMeans(n_clusters=4, random_state=42, n_init='auto')
km_labels = km.fit_predict(X)          # integer label per point

# 2. Dimensionality reduction: PCA → 1D and back (for visualisation)
pca = PCA(n_components=1)
X_1d = pca.fit_transform(X)            # project to 1 principal component
X_reconstructed = pca.inverse_transform(X_1d)  # project back to 2D

# 3. Density estimation: Kernel Density on 1st feature
kde = KernelDensity(bandwidth=0.5, kernel='gaussian')
kde.fit(X[:, :1])                      # fit on first feature only (1D for easy plot)
x_grid = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300).reshape(-1, 1)
log_dens = kde.score_samples(x_grid)   # log-probability at each grid point

# ── Three side-by-side plots ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1 — Clustering result
for cid in np.unique(km_labels):
    mask = km_labels == cid
    axes[0].scatter(X[mask, 0], X[mask, 1],
                    s=25, alpha=0.6, label=f'Cluster {cid}',
                    edgecolors='white', linewidths=0.3)
axes[0].scatter(km.cluster_centers_[:, 0],
                km.cluster_centers_[:, 1],
                marker='X', s=150, c='black', zorder=5, label='Centroids')
axes[0].set_title("Clustering (K-Means)\nQuestion: which group?", fontsize=11)
axes[0].legend(fontsize=8)

# Plot 2 — Dimensionality reduction
axes[1].scatter(X[:, 0], X[:, 1],
                s=20, alpha=0.3, color='grey', label='Original')
axes[1].scatter(X_reconstructed[:, 0], X_reconstructed[:, 1],
                s=20, alpha=0.6, color='crimson', label='PCA projection (1D)')
axes[1].set_title("Dimensionality Reduction (PCA)\nQuestion: what is the main axis?", fontsize=11)
axes[1].legend(fontsize=8)

# Plot 3 — Density estimation
axes[2].hist(X[:, 0], bins=30, density=True,
             alpha=0.4, color='steelblue', label='Data histogram')
axes[2].plot(x_grid, np.exp(log_dens),
             lw=2, color='navy', label='KDE density')
axes[2].set_title("Density Estimation (KDE)\nQuestion: how likely is this point?", fontsize=11)
axes[2].set_xlabel("Feature 1")
axes[2].legend(fontsize=8)

fig.suptitle("Three Different \"Answers\" to the Same Unlabelled Dataset",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## The Evaluation Challenge

In supervised learning, evaluation is straightforward:

1. Hold out a test set with known labels.
2. Predict on the test set.
3. Compare predictions to true labels → accuracy, F1, etc.

In unsupervised learning, **there are no ground-truth labels** to compare against. We use two families of metrics instead:

### Internal metrics (no labels needed)
Measure quality using only the data and the discovered structure.
- **Inertia / Within-cluster sum of squares (WCSS):** total squared distance from each point to its cluster centroid. Lower = tighter clusters.
- **Silhouette score:** for each point, compare its average distance to its own cluster vs. the nearest other cluster. Ranges from −1 (bad) to +1 (perfect). Higher = better.

### External metrics (need labels — used for benchmarking only)
Compare discovered clusters to known ground-truth labels.
- **Adjusted Rand Index (ARI)**
- **Normalised Mutual Information (NMI)**

> In real applications you typically **only have internal metrics**, because if you had labels you'd use supervised learning.


In [ ]:
np.random.seed(42)

# Silhouette score preview — we'll study this in detail in NB05
# km_labels was computed above using sklearn KMeans

sil = silhouette_score(X, km_labels)  # compares intra- vs. inter-cluster distances

print(f"K-Means silhouette score: {sil:.4f}")
print()
print("Interpretation guide:")
print("  Score > 0.7  → strong cluster structure")
print("  Score 0.5–0.7 → reasonable structure")
print("  Score 0.25–0.5 → weak structure (clusters overlap)")
print("  Score < 0.25 → no meaningful structure found")
print()
print("Notice: we computed this WITHOUT ever knowing the true labels.")
print("This is the power of internal evaluation metrics.")


## The Online Retail Dataset — Our Week 11 Running Example

Throughout Week 11, we will use a dataset that mirrors the **UCI Online Retail dataset** (Chen et al., 2012).  
The original contains ~540,000 transactions from a UK-based online gift store, January 2010 – September 2011.

### What the raw data contains
| Column | Description |
|---|---|
| `InvoiceNo` | Unique transaction identifier |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Number of units purchased |
| `InvoiceDate` | Date and time of transaction |
| `UnitPrice` | Price per unit (GBP) |
| `CustomerID` | Unique customer identifier |
| `Country` | Customer's country |

### Why it is perfect for unsupervised learning
- **No natural label column** — there is no "customer type" pre-attached.
- **RFM features** (Recency, Frequency, Monetary value) are standard in retail analytics and produce nicely interpretable clusters.
- **Enough customers** (~4,300 unique IDs) to make clustering interesting but not computationally heavy.
- **Real business value** — the segments we find directly map to marketing actions.

We will work with a **synthetic version** that preserves the statistical properties of the real dataset.


In [ ]:
np.random.seed(42)

# ── Generate synthetic Online Retail customer-level data ──────────────────────
# We simulate RFM (Recency, Frequency, Monetary) features for 500 customers.
# The distributions are chosen to mimic the real UCI Online Retail dataset.

n_customers = 500

# Four latent customer segments (we won't tell the algorithm this)
# Segment A: High spenders, frequent, recent
# Segment B: Moderate spenders, moderate frequency
# Segment C: Low spenders, infrequent, not recent (at-risk)
# Segment D: One-time buyers, very low spend

segment_sizes = [100, 150, 150, 100]  # sum = 500
segment_labels = np.repeat([0, 1, 2, 3], segment_sizes)  # ground truth (hidden)

# TotalSpend: total £ spent across all transactions
spend_params = [(2000, 600), (600, 200), (200, 80), (80, 30)]  # (mean, std) per segment
total_spend = np.concatenate([
    np.random.normal(m, s, n) for (m, s), n in zip(spend_params, segment_sizes)
])
total_spend = np.clip(total_spend, 5, 6000)  # realistic floor and ceiling

# Frequency: number of distinct purchase occasions
freq_params = [(40, 10), (15, 5), (6, 3), (2, 1)]
frequency = np.concatenate([
    np.random.normal(m, s, n) for (m, s), n in zip(freq_params, segment_sizes)
])
frequency = np.clip(frequency, 1, 100).round().astype(int)

# Recency: days since last purchase (lower = more recent = better)
recency_params = [(20, 15), (60, 30), (150, 40), (250, 50)]
recency = np.concatenate([
    np.random.normal(m, s, n) for (m, s), n in zip(recency_params, segment_sizes)
])
recency = np.clip(recency, 1, 365).round().astype(int)

# Shuffle so the segment structure isn't trivially visible by row order
shuffle_idx = np.random.permutation(n_customers)
df_retail = pd.DataFrame({
    'CustomerID': np.arange(1001, 1001 + n_customers)[shuffle_idx],
    'TotalSpend':  np.round(total_spend[shuffle_idx], 2),
    'Frequency':   frequency[shuffle_idx],
    'Recency':     recency[shuffle_idx]
})

print("Shape:", df_retail.shape)
print()
print("First 8 rows:")
print(df_retail.head(8).to_string(index=False))
print()
print("Descriptive statistics:")
print(df_retail.describe().round(2).to_string())


## Week 11 Roadmap

Here is what we will build across all 12 notebooks this week, using the retail dataset as our running thread:

| Notebook | Topic | Retail task |
|---|---|---|
| **NB 01** (this one) | Unsupervised learning intro | Frame the customer segmentation problem |
| **NB 02** | K-Means clustering | Segment customers into k groups |
| **NB 03** | K-Means limitations & alternatives | Handle non-spherical segments |
| **NB 04** | Hierarchical clustering | Build a dendrogram of customer similarity |
| **NB 05** | Cluster evaluation metrics | Choose the right k using silhouette & elbow |
| **NB 06** | DBSCAN | Find dense regions; detect outlier customers |
| **NB 07** | Gaussian Mixture Models | Soft assignment + density estimation |
| **NB 08** | PCA | Compress the feature space; visualise in 2D |
| **NB 09** | t-SNE & UMAP | Non-linear embeddings for visualisation |
| **NB 10** | Autoencoders | Deep dimensionality reduction |
| **NB 11** | Putting it together | Full pipeline: reduce → cluster → profile |
| **NB 12** | Project | Segment real customers; write an insight report |

Each notebook builds on the previous one. By NB 11, you will have a complete, production-ready segmentation pipeline.


In [ ]:
np.random.seed(42)

# ── First look at the retail data: TotalSpend vs. Frequency ──────────────────
# We plot without any colour coding to pose the clustering question visually.

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    df_retail['TotalSpend'],
    df_retail['Frequency'],
    alpha=0.55,
    s=40,
    color='steelblue',
    edgecolors='white',
    linewidths=0.4
)

ax.set_xlabel("Total Spend (£)", fontsize=12)
ax.set_ylabel("Purchase Frequency (# visits)", fontsize=12)
ax.set_title(
    "500 Customers — TotalSpend vs. Frequency\n"
    "Can you spot the natural groups? (No labels provided)",
    fontsize=13
)

# Annotate the motivating question
ax.text(
    0.97, 0.97,
    "Goal: discover customer\nsegments without labels",
    transform=ax.transAxes,
    ha='right', va='top',
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8)
)

plt.tight_layout()
plt.show()


## Self-Check Questions

Test your understanding before moving on. Try to answer each question on your own, then reveal the answer.

---

**Question 1:**  
You have a dataset of 1 million songs with features like tempo, key, and loudness, but **no genre labels**. You want to group similar songs together. Which learning paradigm applies, and why?

<details><summary>Show answer</summary>

**Unsupervised learning** — specifically **clustering**.  
Because there are no genre labels (no y), you cannot use supervised learning. You want to discover natural groupings from the feature matrix X alone. K-Means or DBSCAN would be appropriate starting points.

</details>

---

**Question 2:**  
A colleague claims: "Unsupervised learning is easy to evaluate because you just check how tight the clusters are." What is wrong with this claim?

<details><summary>Show answer</summary>

Tightness (inertia) is only an **internal metric** and has a fundamental flaw: it can always be improved by increasing k (more clusters = tighter). A degenerate solution where every point is its own cluster has zero inertia but no useful structure.  
True evaluation requires balancing compactness against separation (e.g., silhouette score), checking stability across random seeds, and validating that the segments are **interpretable and actionable** — something no single number captures.  
Without ground-truth labels, evaluation is inherently harder than in supervised learning.

</details>

---

**Question 3:**  
In the three-families diagram, which family is most relevant when your goal is to **detect fraudulent credit card transactions** (flagging unusual individual transactions)? Justify your answer.

<details><summary>Show answer</summary>

**Density estimation** is most directly relevant.  
The idea: learn the probability density p(x) of *normal* transactions. A transaction with very low p(x) is flagged as anomalous (fraud).  
Methods like Isolation Forest, One-Class SVM, or a Gaussian Mixture Model fit this paradigm.  
Clustering could also be used (flag points that don't belong to any cluster, as in DBSCAN noise points), so "density estimation" and "clustering" can both apply — but density estimation is the more principled framing for anomaly detection.

</details>


## Key Takeaways

| Concept | Summary |
|---|---|
| **Unsupervised learning** | Learning from data with no labels — find hidden structure |
| **Three families** | Clustering (groups), Dimensionality Reduction (structure), Density Estimation (distribution) |
| **vs. Supervised** | No y in the loss; evaluation is harder; works on raw unlabelled data |
| **Evaluation** | Internal metrics (silhouette, inertia) don't need labels; external metrics (ARI) do |
| **Week 11 dataset** | Synthetic retail RFM data — TotalSpend, Frequency, Recency for 500 customers |

---

**Up next — Notebook 02:** K-Means clustering from scratch.  
We will implement Lloyd's algorithm step by step, visualise every iteration, and compare our implementation against sklearn.

> *"Clustering is not about finding what you put in. It's about discovering what was always there."*
